# Hands-On Lab: Linear Regression from First Principles
Accompanies **Chapter 7: Linear Regression from First Principles**.

In this lab, we:
1. Implement the closed-form Normal Equations $\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$ from scratch using NumPy.
2. Verify numerical equivalence against `scikit-learn`'s `LinearRegression`.
3. Diagnose Gauss-Markov assumptions via Residual Plots (Homoscedasticity vs Heteroscedasticity).
4. Measure $R^2$ and Adjusted $R^2$.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Load peer-to-peer loan data
data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)

# Prepare clean numeric features for regression
df['Interest.Rate'] = pd.to_numeric(df['Interest.Rate'].astype(str).str.replace('%', ''), errors='coerce')
df['Debt.To.Income.Ratio'] = pd.to_numeric(df['Debt.To.Income.Ratio'].astype(str).str.replace('%', ''), errors='coerce')
df['FICO.Range'] = df['FICO.Range'].astype(str).str.split('-').apply(lambda x: (float(x[0]) + float(x[1])) / 2 if isinstance(x, list) and len(x) == 2 else np.nan)

features = ['Amount.Requested', 'FICO.Range', 'Debt.To.Income.Ratio']
target = 'Interest.Rate'

clean_df = df[features + [target]].dropna()
X = clean_df[features].values
y = clean_df[target].values

print(f"X shape: {X.shape}, y shape: {y.shape}")

### 1. From-Scratch Normal Equations Implementation
Recall from Chapter 7:
$$\mathbf{w}^* = (\mathbf{X}_{aug}^T \mathbf{X}_{aug})^{-1} \mathbf{X}_{aug}^T \mathbf{y}$$
where $\mathbf{X}_{aug} = [\mathbf{1}, \mathbf{X}]$.

In [ ]:
# Augment X with column of ones for the intercept
n_samples = X.shape[0]
X_aug = np.column_stack([np.ones(n_samples), X])

# Solve normal equations: w* = (X^T X)^(-1) X^T y
w_star = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)

intercept_custom = w_star[0]
weights_custom = w_star[1:]

print("--- Custom Normal Equations ---")
print(f"Intercept: {intercept_custom:.4f}")
for name, w in zip(features, weights_custom):
    print(f"Weight ({name}): {w:.6f}")

### 2. Verification Against Scikit-Learn

In [ ]:
sk_model = LinearRegression(fit_intercept=True)
sk_model.fit(X, y)

print("--- Scikit-Learn LinearRegression ---")
print(f"Intercept: {sk_model.intercept_:.4f}")
for name, w in zip(features, sk_model.coef_):
    print(f"Weight ({name}): {w:.6f}")

assert np.isclose(intercept_custom, sk_model.intercept_, atol=1e-5)
assert np.allclose(weights_custom, sk_model.coef_, atol=1e-5)
print("\n[SUCCESS] Exact mathematical match between Custom OLS and Scikit-Learn!")

### 3. Residual Diagnostics (Homoscedasticity vs Heteroscedasticity)

In [ ]:
y_pred = sk_model.predict(X)
residuals = y - y_pred

plt.figure(figsize=(9, 5))
plt.scatter(y_pred, residuals, alpha=0.3, color='steelblue', edgecolors='none', s=20)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('Fitted Values ($\\hat{y}$)', fontsize=11)
plt.ylabel('Residuals ($e = y - \\hat{y}$)', fontsize=11)
plt.title('Residuals vs. Fitted Values Diagnostic Plot', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

r2 = r2_score(y, y_pred)
n, p = X.shape
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print(f"R-squared:          {r2:.4f}")
print(f"Adjusted R-squared: {adj_r2:.4f}")
print(f"RMSE:               {np.sqrt(mean_squared_error(y, y_pred)):.4f}")